In [1]:
!nvidia-smi


Mon Jul  6 16:55:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71                 Driver Version: 595.71         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P4              8W /   42W |       0MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [3]:
torch.cuda.device_count()

1

In [5]:
def try_gpu(i=0):  #@save
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():  #@save
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}')
             for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

In [7]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

In [9]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [12]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y.device, Y

(device(type='cpu'),
 tensor([[0.9145, 0.3289, 0.7609],
         [0.8857, 0.9589, 0.8822]]))

In [17]:
Z = Y.cuda()
print(X)
print(Z)
X + Z


tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[0.9145, 0.3289, 0.7609],
        [0.8857, 0.9589, 0.8822]], device='cuda:0')


tensor([[1.9145, 1.3289, 1.7609],
        [1.8857, 1.9589, 1.8822]], device='cuda:0')

In [18]:
Z.cuda(0) is Z

True

In [19]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

In [20]:
net(X)

tensor([[0.0625],
        [0.0625]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [21]:
net[0].weight.data.device

device(type='cuda', index=0)